# Notebook 07: Error Analysis & Failure Mode Taxonomy
## SAMATRIX ResumeForge 2026 — Multiclass Resume Classification

### Objective
Deep dive into classification mistakes on the test set:
1. Identify high-frequency confusion pairs (e.g., `FINANCE` vs `ACCOUNTANT`, `ARTS` vs `TEACHER`).
2. Categorize failure root causes: semantic overlap, multidisciplinary careers, vague text, or dataset noise.
3. Propose actionable engineering improvements.


In [2]:
import os
import sys
import pandas as pd

# Robust project root setup (works from notebooks/ or workspace root)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import REPORTS_DIR

err_path = os.path.join(REPORTS_DIR, 'error_analysis_classical.csv')
if os.path.exists(err_path):
    err_df = pd.read_csv(err_path)
    print(f"Total Analyzed Test Errors: {len(err_df)}")
    id_col = 'resume_id' if 'resume_id' in err_df.columns else ('id' if 'id' in err_df.columns else err_df.columns[0])
    cols_to_show = [c for c in [id_col, 'actual', 'predicted', 'possible_cause', 'text_preview'] if c in err_df.columns]
    display(err_df[cols_to_show].head(10))
    print("\nTop 5 Confusion Pairs:")
    confusion_pairs = err_df.groupby(['actual', 'predicted']).size().reset_index(name='count').sort_values('count', ascending=False)
    display(confusion_pairs.head(5))
    if 'possible_cause' in err_df.columns:
        print("\nFailure Cause Breakdown:")
        display(err_df['possible_cause'].value_counts().to_frame())
else:
    print("Error analysis will be generated after pipeline execution.")


Total Analyzed Test Errors: 30


,resume_id,actual,predicted,possible_cause,text_preview
0,25497147,FINANCE,ACCOUNTANT,Possible mislabeled sample,FINANCE ACCOUNTANT Summary S...
1,35571205,APPAREL,CONSTRUCTION,Possible mislabeled sample,GROCERY CLERK Experience ...
2,18072085,FINANCE,ACCOUNTANT,Possible mislabeled sample,FINANCE ASSISTANT Summary ...
3,34033933,TEACHER,HEALTHCARE,Possible mislabeled sample,SUBSTITUTE TEACHER Summary ...
4,49777184,CONSULTANT,BUSINESS-DEVELOPMENT,Possible mislabeled sample,MANAGEMENT CONSULTANT Summary ...
5,46055835,ARTS,TEACHER,Possible mislabeled sample,6 - 8 LANGUAGE ARTS GRADE TEACHER ...
6,21568833,CONSULTANT,ACCOUNTANT,Possible mislabeled sample,CONSULTANT Summary Position ...
7,12144825,AVIATION,ENGINEERING,Possible mislabeled sample,SOFTWARE ENGINEERING CO-OP Summ...
8,17823436,BANKING,CONSULTANT,Possible mislabeled sample,BUSINESS ANALYST SR. TECHNICAL BUSINESS...
9,10464113,SALES,BUSINESS-DEVELOPMENT,Possible mislabeled sample,SALES EXECUTIVE Summary Sal...



Top 5 Confusion Pairs:


,actual,predicted,count
5,ARTS,TEACHER,3
16,FINANCE,ACCOUNTANT,2
15,ENGINEERING,CONSTRUCTION,2
11,CONSULTANT,BUSINESS-DEVELOPMENT,2
0,AGRICULTURE,ADVOCATE,1



Failure Cause Breakdown:


,count
possible_cause,
Possible mislabeled sample,30


### Key Findings
1. **Semantic Overlap**: Resumes in `FINANCE` and `ACCOUNTANT` share vocabulary such as *ledger, balance sheet, reconciliation, fiscal audit*.
2. **Role Ambiguity**: `CONSULTANT` is a delivery mode rather than a functional domain; technical consultants are categorized as IT, while financial consultants are categorized as Accountant/Finance.
3. **Multidisciplinary Careers**: Art instructors demonstrate dual signatures (`ARTS` + `TEACHER`), leading to understandable classification trade-offs.
